# ✏️ Air Canvas — Draw with Your Pen Cap

**Point a brightly-coloured pen cap toward the camera and draw in the air.**

| Key | Action |
|-----|--------|
| **C** | Recalibrate color |
| **E** | Erase / clear canvas |
| **S** | Save canvas as PNG |
| **Q** | Quit |
| **1–5** | Switch brush colour |
| **[ / ]** | Smaller / Larger brush |

> **Best cap colours:** 🟢 Green · 🟠 Orange · 🔴 Red · 🩷 Pink · 🔵 Blue

In [ ]:
# ── 1. Install dependencies (run once) ────────────────────────
# !pip install opencv-python numpy

In [ ]:
import cv2
import numpy as np
import time
from collections import deque


# ═══════════════════════════════════════════════════════════════
#  CONFIG
# ═══════════════════════════════════════════════════════════════
class Config:
    CALIB_SECONDS   = 4       # seconds to auto-calibrate
    CALIB_BOX_FRAC  = 0.12    # ROI box fraction of frame
    HSV_H_TOLERANCE = 15      # hue tolerance ±
    HSV_S_MIN       = 80
    HSV_V_MIN       = 80
    MIN_CONTOUR_AREA= 150
    CANVAS_ALPHA    = 0.85    # drawing opacity on live feed
    WIN_NAME        = 'Air Canvas'
    COLORS = [
        ('1 Red',    (0,  0,  220)),
        ('2 Blue',   (220,80, 0  )),
        ('3 Green',  (30, 180,30 )),
        ('4 Yellow', (0,  220,220)),
        ('5 White',  (255,255,255)),
    ]


# ═══════════════════════════════════════════════════════════════
#  KALMAN SMOOTHER
# ═══════════════════════════════════════════════════════════════
class KalmanTracker:
    def __init__(self):
        self.kf = cv2.KalmanFilter(4, 2)
        self.kf.measurementMatrix   = np.array([[1,0,0,0],[0,1,0,0]], np.float32)
        self.kf.transitionMatrix    = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], np.float32)
        self.kf.processNoiseCov     = np.eye(4, dtype=np.float32) * 0.03
        self.kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 0.5
        self.kf.errorCovPost        = np.eye(4, dtype=np.float32)
        self.initialized = False

    def update(self, x, y):
        meas = np.array([[np.float32(x)], [np.float32(y)]])
        if not self.initialized:
            self.kf.statePre = np.array([[x],[y],[0],[0]], np.float32)
            self.initialized = True
        self.kf.correct(meas)
        pred = self.kf.predict()
        return int(pred[0][0]), int(pred[1][0])

    def predict_only(self):
        pred = self.kf.predict()
        return int(pred[0][0]), int(pred[1][0])

    def reset(self):
        self.initialized = False


# ═══════════════════════════════════════════════════════════════
#  COLOR CALIBRATOR
# ═══════════════════════════════════════════════════════════════
class ColorCalibrator:
    def __init__(self):
        self.lower = self.upper = None
        self.calibrated = False
        self._samples = []

    def add_sample(self, roi_bgr):
        hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
        self._samples.append(hsv)

    def finalize(self):
        if not self._samples:
            return False
        all_px = np.concatenate([s.reshape(-1,3) for s in self._samples])
        h_med  = np.median(all_px[:,0])
        ht     = Config.HSV_H_TOLERANCE
        self.lower = np.array([max(0,h_med-ht), Config.HSV_S_MIN, Config.HSV_V_MIN], np.uint8)
        self.upper = np.array([min(179,h_med+ht), 255, 255], np.uint8)
        self.calibrated = True
        self._samples = []
        return True

    def mask(self, frame_bgr):
        if not self.calibrated:
            return None
        hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
        m   = cv2.inRange(hsv, self.lower, self.upper)
        k   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,7))
        m   = cv2.morphologyEx(m, cv2.MORPH_OPEN,   k, iterations=1)
        m   = cv2.morphologyEx(m, cv2.MORPH_DILATE, k, iterations=2)
        return m

    def reset(self):
        self.lower = self.upper = None
        self.calibrated = False
        self._samples = []


# ═══════════════════════════════════════════════════════════════
#  STROKE RENDERER
# ═══════════════════════════════════════════════════════════════
class StrokeRenderer:
    def __init__(self, shape):
        h, w = shape[:2]
        self.canvas  = np.zeros((h, w, 3), np.uint8)
        self.prev_pt = None

    def draw(self, pt, color, size):
        if self.prev_pt and pt:
            cv2.line(self.canvas, self.prev_pt, pt, color, size, cv2.LINE_AA)
            cv2.circle(self.canvas, pt, size//2, color, -1, cv2.LINE_AA)
        self.prev_pt = pt

    def lift(self):
        self.prev_pt = None

    def clear(self):
        self.canvas[:] = 0
        self.prev_pt   = None


# ═══════════════════════════════════════════════════════════════
#  MAIN APP
# ═══════════════════════════════════════════════════════════════
class AirCanvas:
    def __init__(self, camera_index=0):
        self.cap = cv2.VideoCapture(camera_index)
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
        self.cap.set(cv2.CAP_PROP_FPS, 60)
        ret, frame = self.cap.read()
        if not ret:
            raise RuntimeError('Cannot open camera.')
        self.h, self.w = frame.shape[:2]
        self.calibrator = ColorCalibrator()
        self.kalman     = KalmanTracker()
        self.renderer   = StrokeRenderer((self.h, self.w))
        self.state      = 'calibrating'
        self.calib_start= time.time()
        self.lost_frames= 0
        self.MAX_LOST   = 8
        self.brush_idx  = 0
        self.brush_color= Config.COLORS[0][1]
        self.brush_size = 6
        self.trail      = deque(maxlen=20)
        self._fps_t     = time.time()
        self._fps_n     = 0
        self._fps       = 0.0

    def _calib_roi(self, frame):
        bx = int(self.w * Config.CALIB_BOX_FRAC)
        by = int(self.h * Config.CALIB_BOX_FRAC)
        cx, cy = self.w//2, self.h//2
        return frame[cy-by:cy+by, cx-bx:cx+bx], (cx-bx, cy-by, cx+bx, cy+by)

    def _find_tip(self, mask):
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts:
            return None
        c = max(cnts, key=cv2.contourArea)
        if cv2.contourArea(c) < Config.MIN_CONTOUR_AREA:
            return None
        return tuple(c[c[:,:,1].argmin()][0])

    def _draw_ui(self, frame, tip_smooth):
        ov = frame.copy()
        # Palette
        bar_y = self.h - 60
        for i, (name, bgr) in enumerate(Config.COLORS):
            x = 10 + i*60
            cv2.rectangle(ov, (x,bar_y), (x+50,bar_y+50), bgr, -1)
            if i == self.brush_idx:
                cv2.rectangle(ov, (x-3,bar_y-3), (x+53,bar_y+53), (255,255,255), 3)
            cv2.putText(ov, name[0], (x+17,bar_y+33), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,0,0), 2)
        # FPS & state
        cv2.putText(ov, f'FPS:{self._fps:.1f}', (10,28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,100), 2)
        st = 'CALIBRATING — hold pen cap in box' if self.state=='calibrating' else 'DRAWING'
        cv2.putText(ov, st, (self.w//2-180, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,220,255), 2)
        # Calib box
        if self.state == 'calibrating':
            _, (x1,y1,x2,y2) = self._calib_roi(frame)
            prog = min((time.time()-self.calib_start)/Config.CALIB_SECONDS, 1.0)
            cv2.rectangle(ov, (x1,y1), (x2,y2), (0,255,255), 3)
            cv2.rectangle(ov, (x1,y2+5), (x1+int((x2-x1)*prog),y2+12), (0,255,100), -1)
        # Crosshair
        if tip_smooth:
            x, y = tip_smooth
            for j, pt in enumerate(self.trail):
                r = max(1, int(self.brush_size*(j/max(len(self.trail),1))*0.6))
                cv2.circle(ov, pt, r, self.brush_color, -1, cv2.LINE_AA)
            cv2.line(ov,(x-18,y),(x+18,y),(255,255,255),1,cv2.LINE_AA)
            cv2.line(ov,(x,y-18),(x,y+18),(255,255,255),1,cv2.LINE_AA)
            cv2.circle(ov,(x,y),self.brush_size+4,(255,255,255),1,cv2.LINE_AA)
            cv2.circle(ov,(x,y),self.brush_size,self.brush_color,-1,cv2.LINE_AA)
        # Brush size
        cv2.circle(ov,(self.w-60,self.h-70),self.brush_size,self.brush_color,-1,cv2.LINE_AA)
        # Help
        cv2.putText(ov,'C=recalib  E=erase  S=save  Q=quit',(10,self.h-120),
                    cv2.FONT_HERSHEY_SIMPLEX,0.47,(180,180,180),1)
        cv2.putText(ov,'1-5=color   [/]=brush size',(10,self.h-100),
                    cv2.FONT_HERSHEY_SIMPLEX,0.47,(180,180,180),1)
        cv2.addWeighted(ov, 0.88, frame, 0.12, 0, frame)

    def run(self):
        print('═'*58)
        print('  AIR CANVAS  |  Point pen cap to camera, hold in BOX')
        print('  C=recalib  E=erase  S=save  Q=quit  1-5=color  [/]=size')
        print('═'*58)
        cv2.namedWindow(Config.WIN_NAME, cv2.WINDOW_NORMAL)
        cv2.resizeWindow(Config.WIN_NAME, min(self.w,1280), min(self.h,720))

        while True:
            ret, frame = self.cap.read()
            if not ret:
                break
            frame = cv2.flip(frame, 1)

            # FPS
            self._fps_n += 1
            now = time.time()
            if now - self._fps_t >= 1.0:
                self._fps = self._fps_n / (now - self._fps_t)
                self._fps_n = 0; self._fps_t = now

            tip_smooth = None

            if self.state == 'calibrating':
                roi, _ = self._calib_roi(frame)
                self.calibrator.add_sample(roi)
                if time.time()-self.calib_start >= Config.CALIB_SECONDS:
                    if self.calibrator.finalize():
                        self.state = 'tracking'
                        self.kalman.reset()
                        print(f'✅ Calibrated — HSV {self.calibrator.lower}–{self.calibrator.upper}')
                    else:
                        self.calib_start = time.time(); self.calibrator.reset()

            elif self.state == 'tracking':
                mask = self.calibrator.mask(frame)
                if mask is not None:
                    tip_raw = self._find_tip(mask)
                    if tip_raw:
                        self.lost_frames = 0
                        tip_smooth = self.kalman.update(*tip_raw)
                        self.trail.append(tip_smooth)
                        self.renderer.draw(tip_smooth, self.brush_color, self.brush_size)
                    else:
                        self.lost_frames += 1
                        if self.lost_frames <= self.MAX_LOST:
                            tip_smooth = self.kalman.predict_only()
                        else:
                            self.renderer.lift(); self.trail.clear()

            # Compose
            disp = frame.copy()
            cmask = cv2.cvtColor(self.renderer.canvas, cv2.COLOR_BGR2GRAY) > 0
            disp[cmask] = (
                disp[cmask]*(1-Config.CANVAS_ALPHA) +
                self.renderer.canvas[cmask]*Config.CANVAS_ALPHA
            ).astype(np.uint8)
            self._draw_ui(disp, tip_smooth)
            cv2.imshow(Config.WIN_NAME, disp)

            key = cv2.waitKey(1) & 0xFF
            if key in (ord('q'), 27):
                break
            elif key == ord('e'):
                self.renderer.clear(); self.trail.clear(); print('🧹 Cleared')
            elif key == ord('c'):
                self.state='calibrating'; self.calib_start=time.time()
                self.calibrator.reset(); self.kalman.reset(); print('🎯 Recalibrating…')
            elif key == ord('s'):
                fn = f'air_canvas_{int(time.time())}.png'
                cv2.imwrite(fn, self.renderer.canvas); print(f'💾 Saved → {fn}')
            elif ord('1') <= key <= ord('5'):
                self.brush_idx = key-ord('1')
                self.brush_color = Config.COLORS[self.brush_idx][1]
                print(f'🖌  {Config.COLORS[self.brush_idx][0]}')
            elif key == ord('['):
                self.brush_size = max(2, self.brush_size-2)
            elif key == ord(']'):
                self.brush_size = min(30, self.brush_size+2)

        self.cap.release()
        cv2.destroyAllWindows()
        print('👋 Done.')


print('✅ All classes loaded — run the next cell to start!')

In [ ]:
# ── 3. LAUNCH  (change camera_index if needed: 0, 1, 2…) ──────
app = AirCanvas(camera_index=0)
app.run()

## 📝 Tips for best tracking accuracy

| Tip | Why it helps |
|-----|-------------|
| Use a **solid, bright-coloured** cap | Easy to segment from background |
| Keep background **plain** (wall, desk) | Reduces false positives |
| Ensure **good even lighting** | Colour constancy for HSV detection |
| Hold pen **vertical / angled up** | Topmost point = sharpest tip detection |
| Press **C** if pen wanders | Re-calibrate to current lighting |
| Draw at **arm's length** from camera | Stable detection distance |
